<h3><b style="color:red">IMPORTANT:</b> Restart your session everytime you open this workshop</h3>

In [ ]:
%pip install -U evaluate rouge_score rouge chromadb unstructured pypandoc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core

# Load epub book

In [ ]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import CrossEncoder

/tmp/ipykernel_1939/121917837.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredEPubLoader


In [ ]:
# Create text splitter
chunk_size = 512
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [ ]:
# TODO: Load document
epub_loader = UnstructuredEPubLoader("/content/docs/monkeys_paw.epub")


In [ ]:
# TODO Split document
chunks = epub_loader.load_and_split(text_splitter=text_splitter)
print(len(chunks))

56


In [ ]:
# TODO Examine chunk
print(chunks[25])

page_content='“I suppose all old soldiers are the same,” said Mrs. White. “The idea of our listening to such nonsense! How could wishes be granted in these days? And if they could, how could two hundred pounds hurt you, father?”

“Might drop on his head from the sky,” said the frivolous Herbert.

“Morris said the things happened so naturally,” said his father, “that you might if you so wished attribute it to coincidence.”' metadata={'source': '/content/docs/monkeys_paw.epub'}


# Create embeddings

In [ ]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"

# Create an embedding function
embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# TODO: Explore embedding model
texts = [ 'Kermit the Frog is a Muppet character created in 1955 and originally performed by Jim Henson. An anthropomorphic green frog, Kermit is the pragmatic everyman protagonist of numerous Muppet productions, most notably as the showrunner and host of the sketch comedy television series The Muppet Show and a featured role on Sesame Street.',
  'big black bug bleeds black blood']

embed_sentence = embed_func(texts)
#print(embed_sentence)
print('======================================')
print(len(embed_sentence))
print(len(embed_sentence[0]))
print(len(embed_sentence[1]))



2
384
384


In [ ]:
# TODO: Prepare the chunks for inserting into Chroma
# Extract texts from chunks into array
texts = [c.page_content for c in chunks]
print(len(texts))
print(texts[20])


56
“As I wished, it twisted in my hand like a snake.”

“Well, I don’t see the money,” said his son as he picked it up and placed it on the table, “and I bet I never shall.”

“It must have been your fancy, father,” said his wife, regarding him anxiously.

He shook his head. “Never mind, though; there’s no harm done, but it gave me a shock all the same.”


In [ ]:
# Create a PK for each chunk of text
ids = [str(uuid4())[:8] for _ in range(len(texts))]
print(len(ids))
print(ids[20])

56
d36c8131


In [ ]:
# TODO: Create ephemeral Chroma client and save chunks
col_name = "monkeys_paw"

# Create a ChromaDB
ch_client = chromadb.Client()

# Delete the col_name collection, perform clean up before populating collection
try:
  ch_client.delete_collection(name=col_name)
except:
  pass


In [ ]:
# Create the collection with an associated embedding function
monkeyspaw_col = ch_client.create_collection(name=col_name, embedding_function=embed_func)

In [ ]:
# Insert the texts and ids into the collection
monkeyspaw_col.add(
    documents=texts,
    ids=ids
)

In [ ]:
# TODO: Print number of documents in collection
print(monkeyspaw_col.count())

56


In [ ]:
# TODO: Query collection
question = "What is the monkey's paw?"
question = "Who is Morris?"

results = monkeyspaw_col.query(
    query_texts=question,
    n_results=10
)

for k,v in results.items():
  print(f'{k}: {v}')


ids: [['13518486', 'da66afda', '6f302608', '8a2567d4', '076a719d', '0172303e', '1b4ad97b', '4ee2b2c8', 'f4108428', '400c55d5']]
embeddings: None
documents: [['“And has anybody else wished?” persisted the old lady.\n\n“The first man had his three wishes. Yes,” was the reply; “I don’t know what the first two were, but the third was for death. That’s how I got the paw.”\n\nHis tones were so grave that a hush fell upon the group.\n\n“If you’ve had your three wishes, it’s no good to you now, then, Morris,” said the old man at last. “What do you keep it for?”', '“I suppose all old soldiers are the same,” said Mrs. White. “The idea of our listening to such nonsense! How could wishes be granted in these days? And if they could, how could two hundred pounds hurt you, father?”\n\n“Might drop on his head from the sky,” said the frivolous Herbert.\n\n“Morris said the things happened so naturally,” said his father, “that you might if you so wished attribute it to coincidence.”', '“I—was asked to ca

In [ ]:
# Retrieve the text by ids
for id in results['ids'][0]:
  doc = monkeyspaw_col.get(ids=id)
  print(doc)


{'ids': ['13518486'], 'embeddings': None, 'documents': ['“And has anybody else wished?” persisted the old lady.\n\n“The first man had his three wishes. Yes,” was the reply; “I don’t know what the first two were, but the third was for death. That’s how I got the paw.”\n\nHis tones were so grave that a hush fell upon the group.\n\n“If you’ve had your three wishes, it’s no good to you now, then, Morris,” said the old man at last. “What do you keep it for?”'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None]}
{'ids': ['da66afda'], 'embeddings': None, 'documents': ['“I suppose all old soldiers are the same,” said Mrs. White. “The idea of our listening to such nonsense! How could wishes be granted in these days? And if they could, how could two hundred pounds hurt you, father?”\n\n“Might drop on his head from the sky,” said the frivolous Herbert.\n\n“Morris said the things happened so naturally,” said his father, “that you might if you so wished attribu

# Cross Encoder
A Cross-Encoder is a model used to determine the semantic similarity, relevance, or relationship between two pieces of text (such as a search query and a document, or a premise and a hypothesis). It embeds 2 pieces of text **together**, allowing full word-by-word cross-attention.

A Bi-Encoder on the other hand embeds 2 pieces of text **independently** into vectors, then calculates similarity (e.g., Cosine Similarity).

In modern RAG application, we combine both bi-encoder and cross-encoder into a two-stage pipeline:

Stage 1: Retrieval (Bi-Encoder): A fast Bi-Encoder searches through millions of documents in a vector database and instantly narrows them down to the top 50 or 100 most likely candidates.

Stage 2: Re-ranking (Cross-Encoder): The slower, highly accurate Cross-Encoder evaluates only those top 50 or 100 candidates side-by-side with the user's query, re-sorting them to ensure the absolute most contextually relevant documents are placed at the very top.

In [ ]:
# Use a model with only a single label for reranking
# Takes about approx 6 minutes to load
cross_encode_model = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(cross_encode_model)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
import numpy as np

In [ ]:
# Extract all text from the result and append to original_order array
original_order = []
for id in results['ids'][0]:
  doc = monkeyspaw_col.get(ids=id)
  original_order.append(doc['documents'][0])


In [ ]:
# Rerank the documents based on the question
reranked_docs = cross_encoder.rank(question, original_order, top_k=3)

for r in reranked_docs:
  print(r)


{'corpus_id': 5, 'score': np.float32(-2.8290133)}
{'corpus_id': 7, 'score': np.float32(-2.8598795)}
{'corpus_id': 0, 'score': np.float32(-3.353796)}


In [22]:
# Print the reordered docs
for r in reranked_docs:
  print(original_order[r['corpus_id']])
  print('--------------')

“I should like to see those old temples and fakirs and jugglers,” said the old man. “What was that you started telling me the other day about a monkey’s paw or something, Morris?”

“Nothing,” said the soldier, hastily. “Leastways nothing worth hearing.”

“Monkey’s paw?” said Mrs. White, curiously.

“Well, it’s just a bit of what you might call magic, perhaps,” said the sergeant-major, offhandedly.
--------------
The old man rose with hospitable haste, and opening the door, was heard condoling with the new arrival. The new arrival also condoled with himself, so that Mrs. White said, “Tut, tut!” and coughed gently as her husband entered the room, followed by a tall, burly man, beady of eye and rubicund of visage.

“Sergeant-Major Morris,” he said, introducing him.
--------------
“And has anybody else wished?” persisted the old lady.

“The first man had his three wishes. Yes,” was the reply; “I don’t know what the first two were, but the third was for death. That’s how I got the paw.”

Hi

# Question and Answer LLM
In this exercise you will implement a question and answer LLM for the 'A Christmas Carol' book that you have chunked and saved.

The workflow is as follows:
1. Assume you ask the following question regarding the book eg. `"Who is Scrooge?"`?
2. Query the relevant context from Chroma with the question or facts from the question.
3. Combine the question and the top 5 context return by Chroma into a prompt
4. Use `google/flan-t5-base` to answer the question.

Look through the FLAN templates in [Github](https://github.com/google-research/FLAN/blob/main/flan/templates.py) and select an appropriate template for this workshop.

Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop.

Use your RAG workflow to answer the provided questions in `questions_for_rag.txt` file.

In [ ]:
# TODO Your code
# Load model
model_name = "google/flan-t5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [51]:
# TODO Your code
#question = "What game are Mr. White and Herbert playing at the start of the story?" # correct
question = "Who brings the monkey's paw to the Whites' house?" # correct
question = "What is Mr. White's first wish?" # correct
#question = "How is the first wish granted?" # wrong
#question = "How many people can have wishes from the paw, and how many wishes does each receive?" # partial correct

In [52]:
# Pre-retrieval - query rewrite
rewrite_prompt = f"{question}\n\nWhat is sentence that verbalizes this data?"
print(rewrite_prompt)

enc_rewrite_prompt = tokenizer(rewrite_prompt, return_tensors='pt')

enc_statement = model.generate(**enc_rewrite_prompt, min_new_tokens=10)

statement = tokenizer.decode(enc_statement[0], skip_special_tokens=True)
print(statement)


What is Mr. White's first wish?

What is sentence that verbalizes this data?
Mr. White's first wish is to be a doctor.


In [59]:
# TODO Retrieval
top_n = 10
results = monkeyspaw_col.query(
    query_texts=statement,
    n_results=top_n
)

In [60]:
# Post retrieval - rerank
original_order = []
for id in results['ids'][0]:
  doc = monkeyspaw_col.get(ids=id)
  original_order.append(doc['documents'][0])

rerank_top_k = 3
reranked_docs = cross_encoder.rank(question, original_order, top_k=rerank_top_k)

for r in reranked_docs:
  print(r)

context = ""
for r in reranked_docs:
  context += original_order[r['corpus_id']]
  print(original_order[r['corpus_id']])
  print('-----------------------------------')

print(context)

{'corpus_id': 0, 'score': np.float32(3.5711682)}
{'corpus_id': 7, 'score': np.float32(1.9428813)}
{'corpus_id': 8, 'score': np.float32(1.7400424)}
Mr. White took the paw from his pocket and eyed it dubiously. “I don’t know what to wish for, and that’s a fact,” he said, slowly. “It seems to me I’ve got all I want.”

“If you only cleared the house, you’d be quite happy, wouldn’t you?” said Herbert, with his hand on his shoulder. “Well, wish for two hundred pounds, then; that ’ll just do it.”
-----------------------------------
“I was to say that ‘Maw and Meggins’ disclaim all responsibility,” continued the other. “They admit no liability at all, but in consideration of your son’s services, they wish to present you with a certain sum as compensation.”

Mr. White dropped his wife’s hand, and rising to his feet, gazed with a look of horror at his visitor. His dry lips shaped the words, “How much?”

“Two hundred pounds,” was the answer.
-----------------------------------
“And what is there 

In [61]:
import numpy as np

scores = np.array([s['score'] for s in reranked_docs])
print(scores)
print(1/(1 + np.exp(-scores)))

[3.5711682 1.9428813 1.7400424]
[0.97264624 0.8746683  0.8506925 ]


In [64]:
#prompt = f'Read this: {context}\n\n{question}\nWhat is the answer? (If it cannot be answered, return "unanswerable")'

prompt = f'Answer based on context:\n\n{context}\n\n{question}'
# CoT prompt
prompt = f"Answer based on context:\n\n{context}\n\n{question}.\n\nLet's think step by step:"

print(prompt)

Answer based on context:

Mr. White took the paw from his pocket and eyed it dubiously. “I don’t know what to wish for, and that’s a fact,” he said, slowly. “It seems to me I’ve got all I want.”

“If you only cleared the house, you’d be quite happy, wouldn’t you?” said Herbert, with his hand on his shoulder. “Well, wish for two hundred pounds, then; that ’ll just do it.”“I was to say that ‘Maw and Meggins’ disclaim all responsibility,” continued the other. “They admit no liability at all, but in consideration of your son’s services, they wish to present you with a certain sum as compensation.”

Mr. White dropped his wife’s hand, and rising to his feet, gazed with a look of horror at his visitor. His dry lips shaped the words, “How much?”

“Two hundred pounds,” was the answer.“And what is there special about it?” inquired Mr. White as he took it from his son, and having examined it, placed it upon the table.

“It had a spell put on it by an old fakir,” said the sergeant-major, “a very h

In [65]:
enc_prompt = tokenizer(prompt, return_tensors='pt')

enc_answer = model.generate(**enc_prompt)

answer = tokenizer.decode(enc_answer[0], skip_special_tokens=True)

print('---------------------')
print(question)
print('------------------')
print(answer)

---------------------
What is Mr. White's first wish?
------------------
To clear the house. To clear the house. To clear the house. To clear the house.


# Optional - Evaluate the answer from you model

Use either Rouge or Bleu to evaluate the answer from your model against those from the text

In [ ]:
# TODO: Evaluate your answer
import evaluate


# Discussion

1. How did your solution perform?
2. Where do you think are the issues?
3. How can you improve it?